# Attention

## Implementation

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        embedding_dim = query.size(-1)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(embedding_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        output = torch.matmul(attention_weights, value)
        return output, attention_weights


def visualize_attention_matrix(attention_weights, output_path, title="Attention Matrix"):
    matrix = attention_weights.detach().numpy()
    plt.figure(figsize=(8, 6))
    plt.imshow(matrix, cmap="viridis")
    plt.colorbar(label="Attention Weight")
    plt.xlabel("Key Position")
    plt.ylabel("Query Position")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

## Demonstration

In [2]:
torch.manual_seed(0)

batch_size = 1
sequence_length = 10
embedding_dim = 16

query = torch.randn(batch_size, sequence_length, embedding_dim)
key = torch.randn(batch_size, sequence_length, embedding_dim)
value = torch.randn(batch_size, sequence_length, embedding_dim)

attention_module = ScaledDotProductAttention(dropout=0.0)

output, attention_weights = attention_module(query, key, value)

print("Scaled Dot Product Attention Without Mask")
print(f"Query Shape: {tuple(query.shape)}")
print(f"Key Shape: {tuple(key.shape)}")
print(f"Value Shape: {tuple(value.shape)}")
print(f"Output Shape: {tuple(output.shape)}")
print(f"Attention Weights Shape: {tuple(attention_weights.shape)}")
print(f"Attention Weights Row Sum: {attention_weights[0, 0].sum().item():.6f}")

causal_mask = torch.tril(torch.ones(sequence_length, sequence_length)).unsqueeze(0)
masked_output, masked_attention_weights = attention_module(query, key, value, mask=causal_mask)

print("\nScaled Dot Product Attention With Causal Mask")
print(f"Output Shape: {tuple(masked_output.shape)}")
print(f"Attention Weights Shape: {tuple(masked_attention_weights.shape)}")
print(f"First Row Non Zero Entries: {int((masked_attention_weights[0, 0] > 0).sum().item())}")
print(f"Last Row Non Zero Entries: {int((masked_attention_weights[0, -1] > 0).sum().item())}")

dropout_module = ScaledDotProductAttention(dropout=0.5)
dropout_module.train()
dropout_output, dropout_attention_weights = dropout_module(query, key, value, mask=causal_mask)

print("\nScaled Dot Product Attention With Dropout")
print(f"Output Shape: {tuple(dropout_output.shape)}")
print(f"Number Of Zeroed Attention Entries: {int((dropout_attention_weights == 0).sum().item())}")

visualize_attention_matrix(masked_attention_weights[0], "attention_matrix.png", title="Causal Scaled Dot Product Attention")
print("\nSaved attention_matrix.png")

Scaled Dot Product Attention Without Mask
Query Shape: (1, 10, 16)
Key Shape: (1, 10, 16)
Value Shape: (1, 10, 16)
Output Shape: (1, 10, 16)
Attention Weights Shape: (1, 10, 10)
Attention Weights Row Sum: 1.000000

Scaled Dot Product Attention With Causal Mask
Output Shape: (1, 10, 16)
Attention Weights Shape: (1, 10, 10)
First Row Non Zero Entries: 1
Last Row Non Zero Entries: 10

Scaled Dot Product Attention With Dropout
Output Shape: (1, 10, 16)
Number Of Zeroed Attention Entries: 68

Saved attention_matrix.png
